In [ ]:
import os
import subprocess
from pathlib import Path
from typing import Any, cast

from anthropic import Anthropic, omit
from anthropic.types import Message, MessageParam, TextBlock
from dotenv import dotenv_values


def initialize_env() -> None:
    env_path = Path.cwd().parent / ".env.template"
    for key, ref in dotenv_values(env_path).items():
        if ref is None:
            continue
        if not os.environ.get(key):
            os.environ[key] = subprocess.run(
                ["op", "read", ref], capture_output=True, text=True, check=True
            ).stdout.strip()


def anthropic_client() -> Anthropic:
    return Anthropic(
        base_url="https://openrouter.ai/api",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )


def model(name: str = "llama") -> str:
    models = {
        "deepseek": "deepseek/deepseek-v4.1-flash",
        "llama": "meta-llama/llama-3.1-8b-instruct",
        "llama-70b": "meta-llama/llama-3.3-70b-instruct",
        "llama-flagship": "meta-llama/llama-4-maverick",
        "mistral": "mistralai/mistral-small-3.1-24b-instruct",
        "haiku": "anthropic/claude-haiku-4.5",
        "qwen": "qwen/qwen-2.5-coder-32b-instruct",
    }
    return models.get(name, models["llama"])


def get_reply(message: Message) -> str:
    text = next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "",
    )
    if not text:
        return f"[no text block; stop_reason={message.stop_reason}, content={message.content!r}]"
    return text


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(
    messages: list[MessageParam],
    system: str | None = None,
    temperature: float = 1.0,
    stop_sequences: list[str] | None = None,
) -> str:
    params: dict[str, Any] = {
        "model": model(),
        "max_tokens": 10000,
        "thinking": {"type": "disabled"},
        "messages": messages,
        "system": system or omit,
        "temperature": temperature,
        "stop_sequences": stop_sequences or omit,
    }
    message = cast(Message, anthropic_client().messages.create(**params))
    return get_reply(message)


def chat_stream(
    messages: list[MessageParam],
    system: str | None = None,
    temperature: float = 1.0,
    stop_sequences: list[str] | None = None,
):
    params: dict[str, Any] = {
        "model": model(),
        "max_tokens": 10000,
        "thinking": {"type": "disabled"},
        "messages": messages,
        "system": system or omit,
        "temperature": temperature,
        "stop_sequences": stop_sequences or omit,
    }
    with anthropic_client().messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(text, end="")
    return get_reply(stream.get_final_message())


# Initialize environment and prepare message list
initialize_env()
messages: list[MessageParam] = []